# Diffusion full / LoRA fine-tuning

## Goal
Run the independent offline latent text-to-image lifecycle: validate image/caption data, train full and LoRA UNets, select on validation, export, reload, generate, and preview publication. The tiny UNet, VAE and CLIP text encoder are random; generated images do not demonstrate quality.

## Setup
Install `python -m pip install -e ".[diffusion,notebooks,publish]"` from the repository. This walkthrough uses CPU fp32, seed 42, 16×16 synthetic fixtures and two epochs. No base-model downloads or Hub uploads occur. Each execution creates a fresh output directory.

In [1]:
from pathlib import Path
import sys, uuid, io, contextlib, os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
from diffusers.utils import logging as diffusers_logging
diffusers_logging.set_verbosity_error()
diffusers_logging.disable_progress_bar()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
from advai_notebooks.diffusion.config import load_config
from advai_notebooks.diffusion.data import load_data, prepare_data
from advai_notebooks.diffusion.pipeline import run
from advai_notebooks.diffusion.model import DiffusionModel
from advai_notebooks.diffusion.evaluate import evaluate
from advai_notebooks.diffusion.publish import publish
RUN_DIR = ROOT / "outputs/diffusion/notebook" / uuid.uuid4().hex[:8]
def quiet(call):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return call()
print({"device": "cpu", "epochs": 2, "training_seed": 42, "resolution": 16})

{'device': 'cpu', 'epochs': 2, 'training_seed': 42, 'resolution': 16}


## Steps
### 1. Validate images, captions and split integrity
Input: `data/sample/diffusion/samples.jsonl` and its eight original PPM images. Check groups, IDs and decoded image duplicates across splits.

In [2]:
full_cfg = load_config(ROOT / "configs/diffusion/smoke.yaml")
full_cfg.output_dir = str(RUN_DIR / "full")
rows, data_summary = load_data(full_cfg)
print(data_summary)

{'rows': 8, 'splits': {'train': 4, 'validation': 2, 'test': 2}, 'groups': 8, 'fingerprint': '8a37a44d4b3ce59a068edf1d6c4e2c0e84e5b5ebd450f23fdfd293d620af6ee7'}


### 2. Train the full UNet
VAE and text encoder remain frozen. Validation denoising MSE uses fixed per-record noise/time draws and VAE mode; it selects the best checkpoint including epoch zero. Test does not select checkpoints.

In [3]:
full_report = quiet(lambda: run(full_cfg))
print({"method": "full", "selected_epoch": full_report["training"]["best_epoch"],
       "baseline_test_mse": full_report["baseline_test"]["metrics"]["denoising_mse"],
       "selected_test_mse": full_report["finetuned_test"]["metrics"]["denoising_mse"]})

{'method': 'full', 'selected_epoch': 2, 'baseline_test_mse': 1.1493373215198517, 'selected_test_mse': 0.9456455558538437}


### 3. Train the LoRA UNet
Only attention adapters update. This fixture uses rank 4 / alpha 8 to exercise native scaling preservation. Its export bundles the frozen initial pipeline for offline loading.

In [4]:
lora_cfg = load_config(ROOT / "configs/diffusion/lora.yaml")
lora_cfg.output_dir = str(RUN_DIR / "lora")
lora_report = quiet(lambda: run(lora_cfg))
print({"method": "lora", "selected_epoch": lora_report["training"]["best_epoch"],
       "baseline_test_mse": lora_report["baseline_test"]["metrics"]["denoising_mse"],
       "selected_test_mse": lora_report["finetuned_test"]["metrics"]["denoising_mse"]})

{'method': 'lora', 'selected_epoch': 2, 'baseline_test_mse': 1.1493373215198517, 'selected_test_mse': 1.1469150483608246}


## Checks
### 4. Reload selected exports and reproduce test results
A smaller denoising loss on two fixtures is an engineering observation, not evidence of better image generation.

In [5]:
models = {}
for name, report, cfg in [("full", full_report, full_cfg), ("lora", lora_report, lora_cfg)]:
    model = quiet(lambda: DiffusionModel.load(report["export"], device="cpu"))
    actual = evaluate(model, prepare_data(rows, model, cfg), cfg, "test")
    assert actual == report["finetuned_test"]
    models[name] = model
print("Full and LoRA reload exactly reproduced the saved test reports.")

Full and LoRA reload exactly reproduced the saved test reports.


### 5. Inspect fixed-seed generated samples
Prompt: `a red square pattern`; seed 123; four DDIM steps. The 16×16 results are enlarged for display only. These random-origin components do not produce reliable semantic images.

In [6]:
from IPython.display import Image as DisplayImage, display
for name, model in models.items():
    generated = model.save_images(["a red square pattern"], RUN_DIR / "preview" / name, seed=123)
    print(name)
    display(DisplayImage(filename=generated["images"][0]["path"], width=256, height=256))

full


lora


### 6. Merge LoRA and preview publication
The merge loads a copy, preserving the original adapter. Publication dry-run validates the native component package and an English card without network access.

In [7]:
merged = quiet(lambda: models["lora"].save(RUN_DIR / "merged", merged=True))
assert quiet(lambda: DiffusionModel.load(merged)).method == "full"
for name, report in [("full", full_report), ("lora", lora_report)]:
    plan = quiet(lambda: publish(report["export"], "your-org/your-diffusion", dry_run=True))
    print({"format": name, "status": plan["status"], "files": len(plan["files"]),
           "private": plan["private"]})

{'format': 'full', 'status': 'dry_run', 'files': 13, 'private': True}
{'format': 'lora', 'status': 'dry_run', 'files': 14, 'private': True}


## Next Steps
Use `configs/diffusion/hf.yaml` with an explicitly supported Stable Diffusion base and split-disjoint real image/caption records. Validate image quality separately. Resume interrupted epochs from the same run's `latest` alias; inference exports lack optimizer/RNG state. CUDA/MPS and real pretrained models need target-hardware verification. See `docs/diffusion.md` for schemas, CLI and current limits. No model or Dataset is published by this notebook.